In [0]:
from pyspark.sql.types import StructType, StructField, StringType

bad_schema_df = spark.createDataFrame(
    [("test_series", "test_area", "2026-01-01", "test_product", "MBBL/D", "not_a_number")],
    ["series_bk", "duoarea_bk", "period_bk", "product_bk", "units", "consumption_value"]
)

try:
    bad_schema_df.write.format("delta").mode("append") \
        .saveAsTable("dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption")
except Exception as e:
    print(f"Write rejected as expected: {type(e).__name__}")
    print(str(e)[:500])

##### Conclusion:
- Delta rejects the write when the incoming column type
- does not match the existing table's column type (period_bk: string vs date).
- This is schema enforcement by default – no mergeSchema or autoMerge
- option was passed, so Delta did not attempt to coerce the type,
- it rejected the entire write immediately.

In [0]:
from pyspark.sql.functions import lit, col

new_column_df = (spark.table("dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption")
    .limit(1)
    .withColumn("data_quality_flag", lit("verified"))
)

try:
    new_column_df.write.format("delta").mode("append") \
        .saveAsTable("dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption")
except Exception as e:
    print(f"Write rejected without mergeSchema: {type(e).__name__}")

In [0]:
new_column_df.write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption")

In [0]:
%sql
DESCRIBE dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption

In [0]:
%sql
SELECT COUNT(*) AS total, COUNT(data_quality_flag) AS non_null_flag
FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption

In [0]:
%sql
DELETE FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
WHERE data_quality_flag = 'verified'

In [0]:
%sql
ALTER TABLE dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);

In [0]:
%sql
ALTER TABLE dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
RENAME COLUMN data_quality_flag TO dq_flag;

In [0]:
%sql
ALTER TABLE dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
DROP COLUMN dq_flag;

In [0]:
spark.sql("DESCRIBE HISTORY dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption").select("version", "timestamp", "operation", "operationParameters").show(20, truncate=False)

In [0]:
from pyspark.sql.types import DecimalType

widened_df = (spark.table("dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption")
    .limit(1)
    .withColumn("consumption_value", col("consumption_value").cast(DecimalType(12, 4)))
)

try:
    widened_df.write.format("delta").mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption")
except Exception as e:
    print(f"Write rejected without type widening enabled: {type(e).__name__}")
    print(str(e)[:300])

In [0]:
%sql
ALTER TABLE dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true')

In [0]:
widened_df.write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption")

In [0]:
%sql
DESCRIBE dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption

In [0]:
%sql
SELECT consumption_sk, COUNT(*) 
FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
GROUP BY consumption_sk
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT * FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
WHERE consumption_sk = 'f99d899d0b3d6991e7ff60f4ed79005817e40fb9ea947b9e5a2c17a41012b422'

In [0]:
%sql
DELETE FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
WHERE consumption_sk = 'f99d899d0b3d6991e7ff60f4ed79005817e40fb9ea947b9e5a2c17a41012b422';

In [0]:
%sql
INSERT INTO dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
VALUES (
  'f99d899d0b3d6991e7ff60f4ed79005817e40fb9ea947b9e5a2c17a41012b422',
  'WDIUPUS2', 'NUS', DATE'2009-01-02', 'EPD0', 'MBBL/D', 
  4318.0000, 'EIA_petroleum_consumption', 
  TIMESTAMP'2026-09-12T06:09:29.074+00:00', NULL
);

In [0]:
%sql
SELECT COUNT(*) FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption;

SELECT consumption_sk, COUNT(*) 
FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
GROUP BY consumption_sk
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT COUNT(*) FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption